In [ ]:
# 2. Use normalized to show the distribution of original data among groups

import seaborn as sns
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

target = 'Normalized_syn'  # minmax_syn

# Construct DataFrame
df = pd.DataFrame({
    'value': list(s[target]) + list(u[target]) + list(c[target]),

    'value_log': (
        list(np.log(s[target])) +
        list(np.log(u[target])) +
        list(np.log(c[target]))
    ),

    'group': (
        ['Syn-HiBiT'] * len(s[target]) +
        ['Syn-HiBiT+Flag-UBL3'] * len(u[target]) +
        ['Syn-HiBiT+Flag-UBL3∆5'] * len(c[target])
    ),

    'group1': (
        list(s['label1']) +
        list(u['label1']) +
        list(c['label1'])
    ),

    # Preserve independent experiment information
    'Experiment_ID': (
        list(s['Experiment_ID']) +
        list(u['Experiment_ID']) +
        list(c['Experiment_ID'])
    ),

    # Correct group labels
    'label': (
        ['SNCA'] * len(s[target]) +
        ['UBL3'] * len(u[target]) +
        ['CAAX'] * len(c[target])
    )
})

df['value_log'] = df['value_log'].replace(-np.inf, 0)


# --------------------------------------------------
# Histogram
# --------------------------------------------------

from matplotlib.lines import Line2D
from matplotlib.patches import Patch

plt.figure(figsize=(7, 5))

ax = sns.histplot(
    data=df,
    x='value',
    hue='group',
    bins=40,
    element='step',
    stat='count',
    palette=['#264a5f', '#d15034', '#f1ae2d'],
    legend=False
)

ax.set_yscale('log')


# Manually create handles and labels
unique_groups = df['group'].unique()

colors = ['#264a5f', '#d15034', '#f1ae2d']

handles = [
    Patch(
        facecolor=colors[i],
        edgecolor='black',
        linewidth=0.5,
        alpha=0.6
    )
    for i in range(len(unique_groups))
]

labels = list(unique_groups)


# Line break handling
new_handles = []
new_labels = []

for h, lab in zip(handles, labels):

    parts = str(lab).split('+')

    line1 = parts[0].strip()
    line2 = '+'.join(parts[1:]).strip() if len(parts) > 1 else ''

    new_handles.append(h)
    new_labels.append(line1)

    if line2:
        empty = Line2D(
            [],
            [],
            linewidth=0,
            marker='',
            linestyle='',
            color='none'
        )

        new_handles.append(empty)
        new_labels.append("  +" + line2)


# Add legend
ax.legend(
    new_handles,
    new_labels,
    title='',
    fontsize=16,
    handlelength=1.5,
    labelspacing=0.2,
    borderpad=0.3,
    frameon=True
)


plt.title('Syn Intensity Histogram', fontsize=18)

plt.xlabel('Normalized intensity', fontsize=16)

plt.ylabel('Log-scaled Pixel count (30 cells)', size=16)

plt.tight_layout()

plt.tick_params(
    axis='both',
    which='major',
    labelsize=16
)
plt.savefig(path+'histogram_threshold_normalize.png')

# Print intensity sum
print(f'SNCA {target} (sum)')
print('SNCA     ', s[target].sum())
print('SNCA+CAAX', c[target].sum())
print('SNCA+UBL3', u[target].sum())


# --------------------------------------------------
# Violin plot + strip plot
# --------------------------------------------------

plt.figure(figsize=(7, 5))

ax = sns.violinplot(
    x="group",
    y="value",
    data=df,
    inner=None,
    color="lightgray"
)

ax = sns.stripplot(
    x="group",
    y="value",
    data=df,
    hue='group',
    dodge=False,
    jitter=0.1,
    alpha=0.01,
    size=1,
    palette=['#264a5f', '#d15034', '#f1ae2d']
)

plt.xlabel('', fontsize=22)

plt.ylabel('Normalized intensity', fontsize=24)

plt.title('Syn Intensity Violinplot', fontsize=24)

plt.xticks(
    rotation=15,
    ha='right',
    fontsize=22
)

plt.tick_params(
    axis='both',
    which='major',
    labelsize=22
)

plt.ticklabel_format(
    style='sci',
    axis='y',
    scilimits=(0, 0)
)

ax.yaxis.get_offset_text().set_fontsize(18)

plt.tight_layout()
plt.savefig(path+'syn_violinplot_stripplot_normalize.png',bbox_inches='tight')